# Getting the discusion about the laws

With this notebook we extract the "discusion en sala" from each law

In [ ]:
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import time
import re

In [15]:
url = "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8446/"
section_key = "1-4"   # the xajaxargs[] value you captured

s = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": url,
    "Origin": "https://www.bcn.cl",
}

r_get = s.get(url, headers=headers, timeout=30)
print("GET status:", r_get.status_code)
print("Cookies after GET:", s.cookies.get_dict())

xajaxr = str(int(time.time() * 1000))  # ms timestamp like your capture

data = [
    ("xajax", "verTextoCompleto"),
    ("xajaxr", xajaxr),
    ("xajaxargs[]", section_key),
]

r_post = s.post(url, headers=headers, data=data, timeout=30)
print("POST status:", r_post.status_code)
print("POST content-type:", r_post.headers.get("Content-Type"))
print("First 200 chars:\n", r_post.text[:200])

root = ET.fromstring(r_post.text)   # parse XML

cmd_nodes = root.findall(".//cmd[@n='js']")
print("Number of <cmd n='js'>:", len(cmd_nodes))

target_js = None
for node in cmd_nodes:
    txt = node.text or ""
    if ".modal-body" in txt and ".html('" in txt:
        target_js = txt
        break

print("Found modal-body injector:", target_js is not None)
print("Preview JS:\n", (target_js[:200] if target_js else "NONE"))

m = re.search(r"\.modal-body'\)\.html\('(.*)'\);\s*$", target_js, flags=re.DOTALL)
if not m:
    raise ValueError("Could not extract modal HTML from JS command.")
raw_html_string = m.group(1)
print("Raw html string length:", len(raw_html_string))

modal_html = raw_html_string
modal_html = modal_html.replace("\\'", "'")
modal_html = modal_html.replace('\\"', '"')
modal_html = modal_html.replace("\\n", "\n")
modal_html = modal_html.replace("\\t", "\t")
modal_html = modal_html.replace("\\r", "\r")
modal_html = modal_html.replace("\\\\", "\\")  # do this last

print(modal_html[:300])

soup = BeautifulSoup(modal_html, "html.parser")

item = soup.select_one("div.item")
print("Found div.item:", item is not None)

# metadata in attributes
metadata = dict(item.attrs) if item else {}
print("Metadata keys:", list(metadata.keys())[:10])

# transcript paragraphs
paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
paragraphs = [p for p in paragraphs if p]  # drop empties
print("Num paragraphs:", len(paragraphs))
print("First 5 paragraphs:")
for p in paragraphs[:5]:
    print("-", p)

GET status: 405
Cookies after GET: {}
POST status: 405
POST content-type: text/html; charset=UTF-8
First 200 chars:
 <!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Human Verification</title>
    <style>
        bo


ParseError: mismatched tag: line 22, column 2 (<string>)

In [8]:
cmd_nodes = root.findall(".//cmd[@n='js']")

for i, node in enumerate(cmd_nodes):
    txt = node.text or ""
    print("\n--- CMD", i, "---")
    print("length:", len(txt))
    print("contains div.item:", "div class=\"item\"" in txt)
    print("preview:", txt[:120].replace("\n", "\\n"))
    


--- CMD 0 ---
length: 26
contains div.item: False
preview: $('.modal-body').html('');

--- CMD 1 ---
length: 74319
contains div.item: True
preview: $('.modal-body').html('<div class="item" fecha="2024-04-03" parteDocumento="http://datos.bcn.cl/recurso/cl/documento/704

--- CMD 2 ---
length: 114
contains div.item: False
preview: $('.modal-title').html('Historia  de la Ley N° 21.788<br> 1. Primer Trámite Constitucional: Cámara de Diputados');

--- CMD 3 ---
length: 25
contains div.item: False
preview: $('#btnTrigger').click();


In [9]:
target_js = None
for node in cmd_nodes:
    txt = node.text or ""
    if "div class=\"item\"" in txt and "$('.modal-body').html(" in txt:
        target_js = txt
        break

if target_js is None:
    raise ValueError("Didn't find the JS command that injects the transcript HTML.")

In [ ]:
start_token = "$('.modal-body').html('"
start = target_js.find(start_token)
if start == -1:
    raise ValueError("Start token not found in target_js.")

start += len(start_token)

end = target_js.rfind("');")
if end == -1 or end <= start:
    raise ValueError("End token not found (or invalid) in target_js.")

raw_html_string = target_js[start:end]
print("Raw html string length:", len(raw_html_string))
print("Raw preview:", raw_html_string[:120])

Raw html string length: 74293
Raw preview: <div class="item" fecha="2024-04-03" parteDocumento="http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-p


In [15]:
modal_html = raw_html_string

# Unescape backslash escapes used inside JS strings
modal_html = modal_html.replace("\\\\", "\\")   # do double backslashes first
modal_html = modal_html.replace("\\'", "'")
modal_html = modal_html.replace('\\"', '"')
modal_html = modal_html.replace("\\n", "\n")
modal_html = modal_html.replace("\\t", "\t")
modal_html = modal_html.replace("\\r", "\r")

print(modal_html[:200])

<div class="item" fecha="2024-04-03" parteDocumento="http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-po1-ds61" refersTo="http://datos.bcn.cl/ontologies/bcn-bills#DiscusionGeneral" ti


In [16]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(modal_html, "html.parser")

item = soup.select_one("div.item")
print("Found div.item:", item is not None)

metadata = dict(item.attrs) if item else {}
print("Metadata keys:", list(metadata.keys()))

paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
paragraphs = [p for p in paragraphs if p]
print("Num paragraphs:", len(paragraphs))
print("First 5 paragraphs:")
for p in paragraphs[:5]:
    print("-", p)

Found div.item: True
Metadata keys: ['class', 'fecha', 'partedocumento', 'refersto', 'tipodocumento', 'tramitacion', 'uridocumento']
Num paragraphs: 323
First 5 paragraphs:
- Fecha 03 de abril, 2024. Diario de Sesión en Sesión 13. Legislatura 372. Discusión General. Se aprueba en general y particular.
- ESTABLECIMIENTO DE NORMAS SOBRE PROTECCIÓN Y FOMENTO DE  ARTESANÍA (PRIMER TRÁMITE CONSTITUCIONAL. BOLETÍN N° 16371-24)
- La señorita  CICARDINI ,   doña Daniella  (Vicepresidenta).-
- Corresponde tratar el proyecto de ley, en primer trámite constitucional, iniciado en mensaje, que dicta normas sobre protección y fomento de la artesanía.
- Para la discusión de este proyecto se otorgarán tres minutos a cada diputada y diputado que se inscriba para hacer uso de la palabra.


## this code discovers if there's a discusion en sala and if it blongs to the cámada de diputados

In [3]:
import re
from datetime import datetime, date
import requests
from bs4 import BeautifulSoup

url = "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8393/"

# Your time window
start_date = date(2022, 3, 11)
end_date   = date(2026, 3, 10)

s = requests.Session()
headers = {"User-Agent": "Mozilla/5.0"}

r = s.get(url, headers=headers, timeout=30)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

# --- 1) Get law number + title (simple, robust enough)
page_text = soup.get_text("\n", strip=True)
m = re.search(r"Historia de la Ley N°\s*([0-9.]+)", page_text)
law_number = m.group(1) if m else None

# Title often appears near the header; you can refine later.
# For now: take the first line after "Historia de la Ley N° ..."
# (If this is messy, we can target the exact HTML element later.)
law_title = None
if law_number:
    idx = page_text.find(f"Historia de la Ley N° {law_number}")
    snippet = page_text[idx: idx + 400] if idx != -1 else ""
    # crude: pick a non-empty line that is not just the header
    lines = [ln.strip() for ln in snippet.split("\n") if ln.strip()]
    # usually title appears a few lines after
    if len(lines) >= 3:
        law_title = lines[2]

print("LAW:", law_number, "-", law_title)

# --- 2) Find the Índice container (heuristic: div containing 'Índice' and many links)
indice_div = None
best = -1
for div in soup.find_all("div"):
    txt = div.get_text(" ", strip=True)
    if "Índice" in txt:
        n_links = len(div.find_all("a"))
        if n_links > best:
            best = n_links
            indice_div = div

if not indice_div:
    raise ValueError("Couldn't locate Índice container reliably.")

links = [a.get_text(" ", strip=True) for a in indice_div.find_all("a")]
links = [" ".join(t.split()) for t in links]  # normalize whitespace

# --- 3) Parse stage lines + subitem lines
stage_line_re = re.compile(r"^(\d+)\.\s+(.*)$")
subitem_re = re.compile(r"^(\d+)\.(\d+)\.\s+(.*?)\s+\((\d{2}-\d{2}-\d{4})\)$")

current_stage = None
current_stage_text = None
current_stage_is_deputies = False

found = []

for t in links:
    # Stage line like: "1. Primer Trámite Constitucional: Cámara de Diputados"
    m_stage = stage_line_re.match(t)
    if m_stage and "Trámite" in t:
        current_stage = int(m_stage.group(1))
        current_stage_text = m_stage.group(2)
        current_stage_is_deputies = ("Cámara de Diputados" in t)
        continue

    # Subitem line like: "1.5. Discusión en Sala (05-03-2025)"
    m_item = subitem_re.match(t)
    if m_item and current_stage is not None:
        s_num = int(m_item.group(1))
        i_num = int(m_item.group(2))
        label = m_item.group(3)
        dstr = m_item.group(4)

        if s_num != current_stage:
            # safety: ignore if it doesn't match our current stage context
            continue

        if "Discusión en Sala" in label:
            d = datetime.strptime(dstr, "%d-%m-%Y").date()
            section_num = f"{s_num}.{i_num}"
            section_key = f"{s_num}-{i_num}"  # for xajaxargs[]
            found.append({
                "stage": current_stage_text,
                "stage_is_deputies": current_stage_is_deputies,
                "section_num": section_num,
                "section_key": section_key,
                "date": d,
                "label": label,
            })

print("\nAll 'Discusión en Sala' found:")
for x in found:
    print("-", x["stage"], "|", x["section_num"], x["date"], "| key:", x["section_key"])

# --- 4) Filter for Cámara de Diputados + date window
valid = [
    x for x in found
    if x["stage_is_deputies"] and (start_date <= x["date"] <= end_date)
]

print("\nVALID (Diputados + in window):")
for x in valid:
    print("✓", x["section_num"], x["date"], "| xajaxargs[] =", x["section_key"], "|", x["stage"])

HTTPError: 405 Client Error: Method Not Allowed for url: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8393/

## Unified pipeline (debug-friendly): discover + fetch + save

This section keeps the process in clear chunks:
1. Configure one law + date window
2. Discover relevant `Discusión en Sala` entries from the index
3. Filter to `Cámara de Diputados` and date window
4. Fetch transcript via AJAX (`xajaxargs[]`)
5. Save transcript locally to avoid repeated scraping

If something breaks, each chunk prints diagnostics to show exactly where and why.

In [1]:
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from datetime import datetime, date
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET

# ========= CONFIG =========
# You can pass either a full URL or a law id.
LAW_URL = "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/"
# LAW_ID = "8393"  # optional alternative

START_DATE = date(2022, 3, 11)
END_DATE = date(2026, 3, 10)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
BASE_RESULTS_DIR = PROJECT_ROOT / "results" / "transcripts"
DEBUG = True

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


def debug_print(*args):
    if DEBUG:
        print(*args)


def normalize_space(text: str) -> str:
    return " ".join((text or "").split())


def ensure_law_url(law_url: str | None = None, law_id: str | None = None) -> str:
    if law_url:
        return law_url
    if law_id:
        return f"https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/{law_id}/"
    raise ValueError("Provide either LAW_URL or law_id.")


print("Config loaded.")
print("LAW_URL:", LAW_URL)
print("Window:", START_DATE, "to", END_DATE)
print("Project root:", PROJECT_ROOT)
print("Save folder:", BASE_RESULTS_DIR)

Config loaded.
LAW_URL: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/
Window: 2022-03-11 to 2026-03-10
Project root: C:\Users\jorge\Documents\Mis documentos\Uni Konstanz\1. ICSS + Python\Project\chilean-polispeech-nlp
Save folder: C:\Users\jorge\Documents\Mis documentos\Uni Konstanz\1. ICSS + Python\Project\chilean-polispeech-nlp\results\transcripts


In [2]:
@dataclass
class DiscussionEntry:
    stage_num: int
    stage_text: str
    stage_is_deputies: bool
    stage_is_constitutional: bool
    stage_is_mixed_chambers: bool
    stage_is_finalization: bool
    section_num: str        # e.g. 1.5
    section_key: str        # e.g. 1-5 (for xajaxargs[])
    label: str
    event_date: date


def _candidate_law_urls(law_url: str) -> list[str]:
    clean = law_url.strip()
    if not clean:
        return []
    a = clean.rstrip("/")
    b = a + "/"
    seen = set()
    out = []
    for u in [clean, a, b]:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def get_page_soup(session: requests.Session, law_url: str, timeout: int = 30, max_attempts: int = 3) -> tuple[BeautifulSoup, str]:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        "Referer": "https://www.bcn.cl/",
        "Upgrade-Insecure-Requests": "1",
    }

    url_candidates = _candidate_law_urls(law_url)
    errors: list[str] = []

    try:
        warm = session.get("https://www.bcn.cl/", headers=headers, timeout=timeout)
        debug_print("Warm-up GET status:", warm.status_code)
    except Exception as exc:
        errors.append(f"warm-up failed: {exc}")

    for candidate_url in url_candidates:
        for attempt in range(1, max_attempts + 1):
            try:
                resp = session.get(candidate_url, headers=headers, timeout=timeout)
                debug_print(f"GET attempt {attempt} [{candidate_url}]:", resp.status_code)

                if resp.status_code in (403, 405, 429, 503):
                    errors.append(f"{candidate_url} -> HTTP {resp.status_code}")
                    if attempt < max_attempts:
                        time.sleep(1.3 * attempt)
                        continue

                resp.raise_for_status()
                return BeautifulSoup(resp.text, "html.parser"), candidate_url

            except Exception as exc:
                errors.append(f"{candidate_url} attempt {attempt}: {exc}")
                if attempt < max_attempts:
                    time.sleep(1.3 * attempt)
                    continue

    msg = " | ".join(errors[-8:])
    raise RuntimeError(
        "Could not fetch law page after retries and URL variants. "
        f"Last diagnostics: {msg}"
    )


def parse_law_metadata(soup: BeautifulSoup) -> dict:
    page_text = soup.get_text("\n", strip=True)
    m = re.search(r"Historia de la Ley N°\s*([0-9.]+)", page_text)
    law_number = m.group(1) if m else None

    title = None
    h1 = soup.find("h1")
    if h1:
        h1_text = normalize_space(h1.get_text(" ", strip=True))
        if h1_text and "Historia de la Ley" not in h1_text:
            title = h1_text

    if not title and law_number:
        anchor = f"Historia de la Ley N° {law_number}"
        idx = page_text.find(anchor)
        snippet = page_text[idx: idx + 1200] if idx != -1 else ""
        lines = [ln.strip() for ln in snippet.split("\n") if ln.strip()]
        for ln in lines[1:15]:
            ln_low = ln.lower()
            if "historia de la ley" in ln_low:
                continue
            if "trámite" in ln_low or "discusión en sala" in ln_low or "mensaje" in ln_low:
                continue
            if len(ln) >= 10:
                title = ln
                break

    return {
        "law_number": law_number,
        "law_title": title,
    }


def find_indice_links(soup: BeautifulSoup) -> list[str]:
    indice_div = None
    best_link_count = -1

    for div in soup.find_all("div"):
        txt = div.get_text(" ", strip=True)
        if "Índice" in txt:
            n_links = len(div.find_all("a"))
            if n_links > best_link_count:
                best_link_count = n_links
                indice_div = div

    if not indice_div:
        raise ValueError("Could not find Índice container.")

    links = [normalize_space(a.get_text(" ", strip=True)) for a in indice_div.find_all("a")]
    links = [x for x in links if x]

    debug_print("Índice links found:", len(links))
    return links


def parse_discussion_entries(index_lines: list[str]) -> list[DiscussionEntry]:
    stage_line_re = re.compile(r"^(\d+)\.\s+(.*)$")
    subitem_re = re.compile(r"^(\d+)\.(\d+)\.\s+(.*?)\s+\((\d{2}-\d{2}-\d{4})\)$")

    entries: list[DiscussionEntry] = []
    current_stage_num = None
    current_stage_text = None
    current_stage_is_deputies = False
    current_stage_is_constitutional = False
    current_stage_is_mixed_chambers = False
    current_stage_is_finalization = False

    for line in index_lines:
        m_stage = stage_line_re.match(line)
        if m_stage and "Trámite" in line:
            current_stage_num = int(m_stage.group(1))
            current_stage_text = m_stage.group(2)

            line_low = line.lower()
            has_deputies = "cámara de diputados" in line_low
            has_senate = "senado" in line_low

            current_stage_is_constitutional = ("trámite constitucional" in line_low)
            current_stage_is_finalization = ("trámite finalización" in line_low) or ("finalización" in line_low)
            current_stage_is_mixed_chambers = has_deputies and has_senate
            current_stage_is_deputies = has_deputies and (not has_senate)
            continue

        m_item = subitem_re.match(line)
        if m_item and current_stage_num is not None:
            s_num = int(m_item.group(1))
            i_num = int(m_item.group(2))
            label = m_item.group(3)
            dstr = m_item.group(4)

            if s_num != current_stage_num:
                continue

            if "Discusión en Sala" in label:
                d = datetime.strptime(dstr, "%d-%m-%Y").date()
                entries.append(
                    DiscussionEntry(
                        stage_num=current_stage_num,
                        stage_text=current_stage_text or "",
                        stage_is_deputies=current_stage_is_deputies,
                        stage_is_constitutional=current_stage_is_constitutional,
                        stage_is_mixed_chambers=current_stage_is_mixed_chambers,
                        stage_is_finalization=current_stage_is_finalization,
                        section_num=f"{s_num}.{i_num}",
                        section_key=f"{s_num}-{i_num}",
                        label=label,
                        event_date=d,
                    )
                )

    return entries


def filter_valid_discussions(entries: list[DiscussionEntry], start_date: date, end_date: date) -> list[DiscussionEntry]:
    return [
        x for x in entries
        if (start_date <= x.event_date <= end_date)
        and x.stage_is_constitutional
        and x.stage_is_deputies
        and (not x.stage_is_mixed_chambers)
        and (not x.stage_is_finalization)
    ]

In [3]:
def discover_law_discussion_candidates(law_url: str, start_date: date, end_date: date) -> dict:
    session = requests.Session()
    soup, resolved_law_url = get_page_soup(session, law_url)

    metadata = parse_law_metadata(soup)
    index_lines = find_indice_links(soup)
    all_discussions = parse_discussion_entries(index_lines)
    valid_discussions = filter_valid_discussions(all_discussions, start_date, end_date)

    print("\n=== DISCOVERY SUMMARY ===")
    print("Resolved URL:", resolved_law_url)
    print("Law number:", metadata.get("law_number"))
    print("Law title :", metadata.get("law_title"))
    print("Total 'Discusión en Sala' found:", len(all_discussions))

    for row in all_discussions:
        tags = []
        tags.append("Constitucional" if row.stage_is_constitutional else "No Constitucional")
        tags.append("Diputados" if row.stage_is_deputies else "No Diputados")
        if row.stage_is_mixed_chambers:
            tags.append("Mixto Dip+Sen")
        if row.stage_is_finalization:
            tags.append("Finalización")
        print(
            f"- {row.section_num} | {row.event_date} | key={row.section_key} | "
            f"[{', '.join(tags)}] | {row.stage_text}"
        )

    print("\nValid (Constitucional + Diputados only + in window):", len(valid_discussions))
    for row in valid_discussions:
        print(f"✓ {row.section_num} | {row.event_date} | xajaxargs[]={row.section_key} | {row.stage_text}")

    return {
        "session": session,
        "law_url": resolved_law_url,
        "metadata": metadata,
        "all_discussions": all_discussions,
        "valid_discussions": valid_discussions,
    }

In [4]:
def extract_modal_html_from_xajax_response(response_text: str) -> str:
    root = ET.fromstring(response_text)
    cmd_nodes = root.findall(".//cmd[@n='js']")

    debug_print("Number of <cmd n='js'> nodes:", len(cmd_nodes))

    # Some responses first clear modal-body with html('') and then inject real HTML.
    # We scan all modal-body html injections and keep the longest non-empty payload.
    html_candidates: list[str] = []

    for node in cmd_nodes:
        txt = node.text or ""
        if ".modal-body" not in txt or ".html(" not in txt:
            continue

        m_single = re.search(r"\.modal-body'\)\.html\('(.*)'\);\s*$", txt, flags=re.DOTALL)
        if m_single:
            html_candidates.append(m_single.group(1))
            continue

        m_double = re.search(r"\.modal-body'\)\.html\(\"(.*)\"\);\s*$", txt, flags=re.DOTALL)
        if m_double:
            html_candidates.append(m_double.group(1))

    if not html_candidates:
        raise ValueError("No JS command found that injects .modal-body HTML.")

    raw_html = max(html_candidates, key=len)

    modal_html = raw_html
    modal_html = modal_html.replace("\\\\", "\\")
    modal_html = modal_html.replace("\\'", "'")
    modal_html = modal_html.replace('\\"', '"')
    modal_html = modal_html.replace("\\n", "\n")
    modal_html = modal_html.replace("\\t", "\t")
    modal_html = modal_html.replace("\\r", "\r")

    return modal_html


def parse_transcript_text_from_modal_html(modal_html: str) -> dict:
    soup = BeautifulSoup(modal_html, "html.parser")
    item = soup.select_one("div.item")
    container = item if item is not None else soup
    attrs = dict(item.attrs) if item else {}

    paragraphs = [p.get_text(" ", strip=True) for p in container.find_all("p")]
    paragraphs = [p for p in paragraphs if p]

    if not paragraphs:
        raw_text = container.get_text("\n", strip=True)
        lines = [normalize_space(ln) for ln in raw_text.split("\n") if normalize_space(ln)]
        cleaned_lines = []
        for ln in lines:
            if cleaned_lines and cleaned_lines[-1] == ln:
                continue
            cleaned_lines.append(ln)
        paragraphs = cleaned_lines

    transcript_text = "\n\n".join(paragraphs)
    return {
        "item_attrs": attrs,
        "paragraphs": paragraphs,
        "transcript_text": transcript_text,
    }


def fetch_discussion_transcript(session: requests.Session, law_url: str, section_key: str, timeout: int = 30, max_attempts: int = 3) -> dict:
    headers_base = {
        "User-Agent": USER_AGENT,
        "Referer": law_url,
        "Origin": "https://www.bcn.cl",
        "X-Requested-With": "XMLHttpRequest",
    }

    last_error = None

    for attempt in range(1, max_attempts + 1):
        pre_get = session.get(law_url, headers={"User-Agent": USER_AGENT}, timeout=timeout)
        debug_print(f"Attempt {attempt}: pre-GET status =", pre_get.status_code)

        xajaxr = str(int(time.time() * 1000))
        payload = [
            ("xajax", "verTextoCompleto"),
            ("xajaxr", xajaxr),
            ("xajaxargs[]", section_key),
        ]

        resp = session.post(law_url, headers=headers_base, data=payload, timeout=timeout)
        debug_print(f"Attempt {attempt}: POST status =", resp.status_code)
        debug_print(f"Attempt {attempt}: POST content-type =", resp.headers.get("Content-Type"))

        if resp.status_code == 405:
            last_error = RuntimeError("Received HTTP 405 from AJAX endpoint.")
            if attempt < max_attempts:
                time.sleep(1.2 * attempt)
                continue
            raise last_error

        try:
            resp.raise_for_status()
            modal_html = extract_modal_html_from_xajax_response(resp.text)
            parsed = parse_transcript_text_from_modal_html(modal_html)

            if len(parsed.get("paragraphs", [])) == 0:
                debug_print("Modal HTML preview (empty parse):", modal_html[:600])

            return {
                "section_key": section_key,
                "xajaxr": xajaxr,
                "raw_response": resp.text,
                "modal_html": modal_html,
                **parsed,
            }
        except Exception as exc:
            last_error = exc
            if attempt < max_attempts:
                time.sleep(1.0 * attempt)
                continue
            raise

    if last_error:
        raise last_error

    raise RuntimeError("Unexpected fetch failure without explicit exception.")

In [5]:
def save_transcript_bundle(base_dir: Path, law_meta: dict, chosen_entry: DiscussionEntry, transcript_payload: dict) -> dict:
    law_number = (law_meta.get("law_number") or "unknown_law").replace(".", "_")
    law_folder = base_dir / f"law_{law_number}"
    law_folder.mkdir(parents=True, exist_ok=True)

    stamp = chosen_entry.event_date.isoformat()
    key = chosen_entry.section_key.replace("-", "_")

    txt_path = law_folder / f"discussion_{stamp}_{key}.txt"
    json_path = law_folder / f"discussion_{stamp}_{key}.json"

    txt_path.write_text(transcript_payload["transcript_text"], encoding="utf-8")

    serializable = {
        "law_number": law_meta.get("law_number"),
        "law_title": law_meta.get("law_title"),
        "section_num": chosen_entry.section_num,
        "section_key": chosen_entry.section_key,
        "stage_text": chosen_entry.stage_text,
        "event_date": chosen_entry.event_date.isoformat(),
        "item_attrs": transcript_payload.get("item_attrs"),
        "paragraph_count": len(transcript_payload.get("paragraphs", [])),
        "transcript_text": transcript_payload.get("transcript_text", ""),
    }
    json_path.write_text(json.dumps(serializable, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "txt_path": txt_path,
        "json_path": json_path,
    }


def save_debug_bundle(base_dir: Path, law_meta: dict, attempts: list[dict], reason: str) -> Path:
    law_number = (law_meta.get("law_number") or "unknown_law").replace(".", "_")
    law_folder = base_dir / f"law_{law_number}"
    law_folder.mkdir(parents=True, exist_ok=True)

    debug_path = law_folder / "debug_no_transcript.json"
    payload = {
        "reason": reason,
        "law_number": law_meta.get("law_number"),
        "law_title": law_meta.get("law_title"),
        "attempt_count": len(attempts),
        "attempts": attempts,
    }
    debug_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return debug_path


def run_pipeline_for_one_law(law_url: str, start_date: date, end_date: date, base_dir: Path) -> dict:
    print("\n================ PIPELINE START ================")

    try:
        discovery = discover_law_discussion_candidates(law_url, start_date, end_date)
    except Exception as exc:
        print("Discovery failed (likely temporary BCN blocking / HTTP 405).")
        print("Error:", exc)
        print("================ PIPELINE END =================\n")
        return {
            "status": "discovery_blocked",
            "error": str(exc),
        }

    valid = discovery["valid_discussions"]

    if not valid:
        print("No target discussion found under strict rule:")
        print("- Stage must be Trámite Constitucional")
        print("- Stage must be Cámara de Diputados only (not mixed with Senado)")
        print("- Stage must not be Trámite finalización")
        print("- Date must be inside selected window")
        print("================ PIPELINE END =================\n")
        return {
            "status": "no_target",
            "discovery": discovery,
        }

    candidates = sorted(valid, key=lambda x: (x.stage_num, x.event_date, x.section_num))

    chosen = None
    transcript = None
    attempt_logs: list[dict] = []

    for candidate in candidates:
        print(
            f"\nTrying candidate {candidate.section_num} | {candidate.event_date} | "
            f"xajaxargs[]={candidate.section_key} | {candidate.stage_text}"
        )
        try:
            payload = fetch_discussion_transcript(
                session=discovery["session"],
                law_url=discovery["law_url"],
                section_key=candidate.section_key,
            )
            paragraph_count = len(payload.get("paragraphs", []))
            print("Paragraphs extracted:", paragraph_count)

            attempt_logs.append({
                "section_key": candidate.section_key,
                "section_num": candidate.section_num,
                "event_date": candidate.event_date.isoformat(),
                "stage_text": candidate.stage_text,
                "paragraph_count": paragraph_count,
                "raw_response_preview": (payload.get("raw_response", "")[:800]),
            })

            if paragraph_count == 0:
                continue

            chosen = candidate
            transcript = payload
            break

        except Exception as exc:
            msg = str(exc)
            print("Fetch failed:", msg)
            attempt_logs.append({
                "section_key": candidate.section_key,
                "section_num": candidate.section_num,
                "event_date": candidate.event_date.isoformat(),
                "stage_text": candidate.stage_text,
                "error": msg,
            })

    if chosen is None or transcript is None:
        debug_path = save_debug_bundle(
            base_dir=base_dir,
            law_meta=discovery["metadata"],
            attempts=attempt_logs,
            reason="No non-empty transcript payload for any valid candidate",
        )
        print("No transcript content returned by BCN for valid candidates.")
        print("Saved debug diagnostics:", debug_path)
        print("================ PIPELINE END =================\n")
        return {
            "status": "no_transcript_content",
            "discovery": discovery,
            "attempts": attempt_logs,
            "debug_path": debug_path,
        }

    print("\nChosen entry:", chosen.section_num, chosen.event_date, "xajaxargs[]=", chosen.section_key)
    print("Transcript preview:\n", transcript["transcript_text"][:600])

    saved = save_transcript_bundle(
        base_dir=base_dir,
        law_meta=discovery["metadata"],
        chosen_entry=chosen,
        transcript_payload=transcript,
    )

    print("\nSaved TXT :", saved["txt_path"])
    print("Saved JSON:", saved["json_path"])
    print("================ PIPELINE END =================\n")

    return {
        "status": "ok",
        "discovery": discovery,
        "chosen_entry": chosen,
        "transcript": transcript,
        "saved": saved,
        "attempts": attempt_logs,
    }

In [6]:
# === RUN (single law) ===
# Change LAW_URL in config cell and re-run from top of the unified section.

result = run_pipeline_for_one_law(
    law_url=ensure_law_url(LAW_URL),
    start_date=START_DATE,
    end_date=END_DATE,
    base_dir=BASE_RESULTS_DIR,
)

# Optional quick access objects
if result.get("status") == "ok":
    chosen_entry = result["chosen_entry"]
    saved_paths = result["saved"]
else:
    chosen_entry = None
    saved_paths = None
    print("Pipeline status:", result.get("status"))


================ PIPELINE START ================
Warm-up GET status: 200
GET attempt 1 [https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/]: 405
GET attempt 2 [https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/]: 405
GET attempt 3 [https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/]: 405
GET attempt 1 [https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035]: 405
GET attempt 2 [https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035]: 405
GET attempt 3 [https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035]: 405
Discovery failed (likely temporary BCN blocking / HTTP 405).
Error: Could not fetch law page after retries and URL variants. Last diagnostics: https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/ -> HTTP 405 | https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/ -> HTTP 405 | https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8035/ -> HTTP 405 | https://www.bcn.cl/historiadelaley/nc/historia-de-l

In [21]:
import requests

section_uri = "http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-p"

for ext in ["html", "rdf", "n3", "ntriples", "json"]:
    u = f"{section_uri}/datos.{ext}"
    r = requests.get(u, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    print(ext, r.status_code, r.headers.get("Content-Type"))
    print(r.text[:120], "\n")

html 200 application/xhtml+xml
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML+RDFa 1.1//EN" "http://www.w3.org/MarkUp 

rdf 200 application/rdf+xml
	xmlns:rdfs="h p://www.w3.org/1999/02/22-rdf-syntax-ns#"

n3 200 text/rdf+n3
  Empty TURTLE

ntriples 200 text/plain
  Empty NT

json 200 application/rdf+json
  }



In [22]:
import requests
from pathlib import Path
import re

endpoint = "https://datos.bcn.cl/sparql"
section_uri = "http://datos.bcn.cl/recurso/cl/documento/704544/seccion/akn704544-po1-ds61"

query = f"""
PREFIX bcnres: <http://datos.bcn.cl/ontologies/bcn-resources#>
SELECT ?xml WHERE {{
  <{section_uri}> bcnres:descripcionXML ?xml .
}}
LIMIT 1
"""

r = requests.get(
    endpoint,
    params={"query": query, "format": "application/sparql-results+json"},
    headers={"Accept": "application/sparql-results+json", "User-Agent": "Mozilla/5.0"},
    timeout=30,
)
r.raise_for_status()
data = r.json()

rows = data["results"]["bindings"]
print("rows:", len(rows))

if not rows:
    print("No descripcionXML found for this section.")
else:
    xml_text = rows[0]["xml"]["value"]
    print("xml preview:", xml_text[:200])

    # VERY simple tag strip (good enough to start; you can improve later)
    plain = re.sub(r"<[^>]+>", "", xml_text)
    plain = re.sub(r"\s+\n", "\n", plain).strip()

    outdir = Path("datos_dump")
    outdir.mkdir(exist_ok=True)
    (outdir / "section_text.txt").write_text(plain, encoding="utf-8")
    print("Saved:", outdir / "section_text.txt")

rows: 1
xml preview:  <debateSection xmlns:bcn="http://datos.bcn.cl" bcn:uriProyectoLey="http://datos.bcn.cl/recurso/cl/proyecto-de-ley/16371-24" bcn:uriResultadoDebate="http://datos.bcn.cl/ontologies/bcn-resources#seApru
Saved: datos_dump\section_text.txt


In [25]:
rows

[{'xml': {'type': 'literal',
   'value': ' <debateSection xmlns:bcn="http://datos.bcn.cl" bcn:uriProyectoLey="http://datos.bcn.cl/recurso/cl/proyecto-de-ley/16371-24" bcn:uriResultadoDebate="http://datos.bcn.cl/ontologies/bcn-resources#seApruebaEnGeneralYParticular" bcn:uriTramiteConstitucional="http://datos.bcn.cl/ontologies/bcn-bills#PrimerTramiteConstitucional" bcn:uriTramiteReglamentario="http://datos.bcn.cl/ontologies/bcn-bills#DiscusionGeneral" id="akn704544-po1-ds61" name="ProyectoDeLey" xmlns="http://www.akomantoso.org/2.0">\n          <heading id="akn704544-po1-ds61-hd14">ESTABLECIMIENTO DE NORMAS SOBRE PROTECCIÓN Y FOMENTO DE  ARTESANÍA (PRIMER TRÁMITE CONSTITUCIONAL. BOLETÍN N° 16371-24)</heading>\n          <p id="akn704544-po1-ds61-p723"> </p>\n          <debateSection id="akn704544-po1-ds61-ds62" name="Antecedente"><p id="akn704544-po1-ds61-ds62-p724">La señorita  CICARDINI ,   doña Daniella  (Vicepresidenta).-</p><p id="akn704544-po1-ds61-ds62-p725">Corresponde tratar el